Task 4: Spark Performance Analysis

In [1]:
# Importing the required libraries for pyspark
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, hash, rand
from pyspark.sql.functions import count, when

Spark Session Configuration

In [2]:
# Create Spark session with resource configuration
spark = SparkSession.builder \
    .appName("HIGGS") \
    .master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .config("spark.default.parallelism", "8") \
    .getOrCreate()

print("Spark Session Created Successfully")

# The Spark application was configured with local[*] to use all available CPU cores. 
# The driver memory was set to 8 GB to process the HIGGS dataset efficiently since my system RAM is 16GB. 
# The shuffle partitions and default parallelism were both set to 8 to improve parallel processing.

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/26 18:10:48 WARN Utils: Your hostname, Sais-MacBook-Pro.local, resolves to a loopback address: 127.0.0.1; using 10.223.134.171 instead (on interface en0)
26/06/26 18:10:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/26 18:10:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/26 18:10:49 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark Session Created Successfully


Load the HIGGS Dataset

In [3]:
# Load transaction dataset from CSV file
df = spark.read.csv("HIGGS.csv.gz", header=False, inferSchema=True)
print("Dataset loaded successfully")

# Assign column names to the dataframe
cols = ["label"] + [f"c{i}" for i in range(1, 29)]
df = df.toDF(*cols)

# Display the first 5 rows
df.show(5)

26/06/26 18:13:19 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Dataset loaded successfully
+-----+------------------+-------------------+-------------------+------------------+-------------------+------------------+--------------------+-------------------+------------------+------------------+-------------------+-------------------+------------------+------------------+--------------------+-------------------+-----------------+-------------------+--------------------+--------------------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+
|label|                c1|                 c2|                 c3|                c4|                 c5|                c6|                  c7|                 c8|                c9|               c10|                c11|                c12|               c13|               c14|                 c15|                c16|              c17|                c18|                 c19|                 c20|              c2

Repartitioning

In [4]:
# Display current number of partitions
print("No of Partitions before repartitioning:", df.rdd.getNumPartitions())

# Repartition the dataset for better parallel processing
df = df.repartition(8)

# Display updated number of partitions
print("No of Partitions after repartitioning:", df.rdd.getNumPartitions())

# The dataset was repartitioned into 8 partitions to distribute the workload evenly across all the available CPU cores. 
# This helps improve parallel execution and resource utilisation effectively.

No of Partitions before repartitioning: 1


[Stage 3:>                                                          (0 + 1) / 1]

No of Partitions after repartitioning: 8


Caching the Dataset

In [5]:
# Cache the dataset in memory for better performance
df.cache()

# Trigger caching by performing an action
df.count()

print("Dataset cached successfully")

# Justification
# The dataset was cached because it is used multiple times during data processing. 
# Caching stores the data in memory, hence reducing repeated computations and improving execution performance. 

[Stage 6:==============>                                            (2 + 6) / 8]

Dataset cached successfully


Execute a Spark Job

In [6]:
# Run a simple Spark program for label count analysis to generate execution details in Spark UI
df.groupBy("label").count().show()

+-----+-------+
|label|  count|
+-----+-------+
|  1.0|5829123|
|  0.0|5170877|
+-----+-------+



Spark UI Interpretation - http://localhost:4040

In [7]:
# The Spark UI was used to monitor the execution of Spark jobs and stages. 
# It shows how Spark divides the application into stages and tasks, helps to analyse job execution, task progress and overall performance.

Spark UI Screenshots

Spark Jobs Tab

![Spark Jobs](Jobs.jpg)

Stages Tab

![Spark Stages](Stages.jpg)

Executors Tab

![Spark Executors](Executors.jpg)

SQl / DataFrame Tab

![Spark SQL/DataFrame](SQL.jpg)